In [25]:
# import required libraries

import torch
import torch.nn as nn
import torch.optim as optim

In [26]:
class Encoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()

        self.fc1 = nn.Linear(28*28, 256)
        self.relu = nn.ReLU()

        self.fc_mu = nn.Linear(256, latent_dim)
        self.log_var = nn.Linear(256, latent_dim)

    def forward(self, x):
        # Flatten image

        x = x.view(x.size(0), -1)
        h = self.relu(self.fc1(x))

        mu = self.fc_mu(h)
        log_var = self.log_var(h)

        return mu, log_var
    
    

In [27]:
def reparameterized(mu, log_var):
        std = torch.exp(0.5 * log_var) # compute standared deviation
        eps = torch.randn_like(std) # eps is randomness 
        return mu + std * eps # sampling

In [28]:
class Decoder(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()

        self.fc1 = nn.Linear(latent_dim, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, 28*28)
        self.sigmoid = nn.Sigmoid() # activation function to squashes any number between 0 to 1

    def forward(self, z):
        h = self.relu(self.fc1(z))
        x_recon = self.sigmoid(self.fc2(h))
        return x_recon

In [29]:
class VAE(nn.Module):
    def __init__(self, latent_dim=2):
        super().__init__()
        
        self.encoder = Encoder(latent_dim)
        self.decoder = Decoder(latent_dim)
        
    def forward(self, x):
        mu, log_var = self.encoder(x)
        z = reparameterized(mu, log_var)
        x_recon = self.decoder(z)
        
        return x_recon, mu, log_var
    
    
    
def vae_loss(recon_x, x, mu, log_var):
    
        # Flatten original image
        x = x.view(x.size(0), -1)
    
        # Reconstruction loss
        recon_loss = nn.functional.mse_loss(recon_x, x, reduction='sum')
    
        # KL divergence
        kl = -0.5 * torch.sum(1 + log_var - mu.pow(2) - log_var.exp())
        return recon_loss + kl

latent_dim = 2
model = VAE(latent_dim)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

x = torch.randn(16, 1, 28, 28)
epochs = 5

for epoch in range(epochs):
    recon, mu, log_var = model(x)
    loss = vae_loss(recon, x, mu, log_var)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    print(f"Epoch {epoch+1}, Loss: {loss.item()}")

with torch.no_grad():
    z = torch.randn(4, latent_dim)
    generated = model.decoder(z)
    
print(generated.shape)

Epoch 1, Loss: 15541.3642578125
Epoch 2, Loss: 15324.1474609375
Epoch 3, Loss: 15111.568359375
Epoch 4, Loss: 14860.630859375
Epoch 5, Loss: 14501.0966796875
torch.Size([4, 784])
